# "Who is Harry Potter?" — Approximate Unlearning for Qwen 2.5 3B

This notebook runs the full unlearning pipeline:
1. Install dependencies & clone repo
2. Download RWKU data & build forget corpus
3. Generate anchor terms
4. Train reinforced model (LoRA)
5. Generate alternative labels
6. Train unlearned model (LoRA)
7. Evaluate on RWKU benchmark

**Requirements:** Google Colab with T4 or A100 GPU.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Clone repo and install dependencies
# Update the URL below to your repo
!git clone https://github.com/YOUR_USERNAME/who-is-harry-potter.git
%cd who-is-harry-potter
!pip install -q -r requirements.txt

In [ ]:
# Cell 3: Step 1 — Download RWKU data & build forget corpus
!python 1_build_forget_corpus.py

In [ ]:
# Cell 4: Step 2 — Generate and save anchor terms
!python 2_anchors.py

In [ ]:
# Cell 5: Quick sanity check — inspect forget corpus and anchors
import json

with open('data/forget_corpus.json') as f:
    corpus = json.load(f)
print(f'Forget corpus: {len(corpus)} passages')
print(f'First passage preview: {corpus[0][:200]}...')

with open('data/anchors.json') as f:
    anchors = json.load(f)
print(f'\nAnchors: {len(anchors)} mappings')
for k, v in list(anchors.items())[:5]:
    print(f'  {k} → {v}')

In [ ]:
# Cell 6: Step 3 — Train reinforced model (LoRA fine-tune on forget corpus)
# This takes ~15-30 min on T4, ~5-10 min on A100
!python 3_train_reinforced.py

In [ ]:
# Cell 7: Step 4 — Generate alternative labels
# This runs both baseline and reinforced models in inference
!python 4_generate_labels.py

In [ ]:
# Cell 8: Step 5 — Train unlearned model
!python 5_train_unlearn.py

In [ ]:
# Cell 9: Step 6 — Evaluate on RWKU benchmark
!python 6_evaluate_rwku.py

In [ ]:
# Cell 10: Quick qualitative check — ask the unlearned model about JKR
from utils import load_tokenizer, load_unlearned_model
import torch

tokenizer = load_tokenizer()
model = load_unlearned_model()
model.eval()

prompts = [
    "Who is J.K. Rowling?",
    "What is Harry Potter about?",
    "Tell me about Hogwarts.",
    "What is the capital of France?",  # control question
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {prompt}')
    print(f'A: {response}\n')